In [0]:
import mlflow
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
from mlflow import MlflowClient
import joblib
import pandas as pd

In [0]:
catalog_name = "supply_chain_daily_demand"
schema_name = "ml_model"
table_name = "testing_data"
model_path =  "models:/supply_chain_daily_demand.ml_model.random_forest_demand/1"
output_table_name = "daily_demand_predictions"
encoder_url = 'models:/m-ee138fd6f7864ab3bc2712a5b86cc3e7'

In [0]:
client = MlflowClient()
model_name = "supply_chain_daily_demand.ml_model.random_forest_demand"
model_version = client.get_model_version(name=model_name,version=1)
run_id = model_version.run_id

In [0]:
class ModelPredictions:
    def load_data(self):
        table_path = f"{catalog_name}.{schema_name}.{table_name}"

        self.data = spark.read.table(table_path)

    def encode_categories(self):
        self.load_data()

        encoder_path = client.download_artifacts(run_id=run_id,path="encoder/encoder.pkl")
        encoder = joblib.load(encoder_path)

        categorical_cols = ["product_name", "destination_city"]

        categorical_data = self.data.select(["feature_uid"] + categorical_cols).toPandas()

        encoded_data = encoder.transform(categorical_data[categorical_cols])

        encoded_cols = encoder.get_feature_names_out(categorical_cols)
        encoded_df = pd.DataFrame(encoded_data,columns=encoded_cols)
        encoded_df["encoded_feature_uid"] = categorical_data["feature_uid"]
        encoded_spark_df = spark.createDataFrame(encoded_df)
        
        encoded_final_df = self.data.join(encoded_spark_df, self.data.feature_uid  == encoded_spark_df.encoded_feature_uid, "inner")\
                               .drop("encoded_feature_uid", "product_name", "destination_city")

        columns = encoded_final_df.columns
        cleaned_columns = []
        for column in columns:
            cleaned_columns.append(column.replace(" ", "_"))

        encoded_final_df = encoded_final_df.toDF(*cleaned_columns)
        return encoded_final_df

    def convert_to_pandas(self, encoded_final_df):
        X_test = encoded_final_df
        X_test = X_test.drop("feature_uid", "total_demand", "transaction_date")
        y_test = encoded_final_df.select("total_demand")

        X_test = X_test.toPandas()
        y_test = y_test.toPandas()

        y_test = y_test["total_demand"]

        return (X_test, y_test)
    
    def predict_demand(self):
        encoded_final_df = self.encode_categories()
        X_test, y_test = self.convert_to_pandas(encoded_final_df)

        model = mlflow.pyfunc.load_model(model_path)

        predictions = model.predict(X_test)

        print("MAE:", mean_absolute_error(predictions, y_test))
        print("MSE:", mean_squared_error(predictions, y_test))
        print("R2:", r2_score(predictions, y_test))

        output_table_path = f"{catalog_name}.{schema_name}.{output_table_name}"
        output_data = self.data.select("feature_uid", "transaction_date", "product_name", "destination_city", "total_demand").toPandas()

        output_data["predicted_demand"] = predictions

        output_data = spark.createDataFrame(output_data)               

        output_data.write.format("delta")\
                         .mode("overwrite")\
                         .saveAsTable(output_table_path)

        print(f"Table loaded to the path: {output_table_path}")

In [0]:
model_pred = ModelPredictions()
model_pred.predict_demand()

In [0]:
%sql
SELECT * FROM supply_chain_daily_demand.ml_model.daily_demand_predictions;